# RAG con Gemini API — Prototipo Asistente ANH

Este notebook adapta el pipeline RAG (chunking → embeddings → Chroma → retrieval → generación) para usar el LLM **como servicio externo vía API** (Gemini 2.5 Flash), en lugar de un modelo local.

Etapas:
1. Base de conocimiento (documentos de ejemplo / PDF normativo).
2. Chunking.
3. Embeddings (HuggingFace, multilingüe).
4. Base vectorial (Chroma).
5. Retrieval.
6. Generación con Gemini API.
7. RAG completo con trazabilidad a la fuente.


## 0. Instalar dependencias

In [ ]:
%pip install -qU \
    langchain-core \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-chroma \
    langchain-community \
    sentence-transformers \
    chromadb \
    pypdf \
    google-generativeai


## 1. Configurar la API Key de Gemini

Consigue una clave gratuita en https://aistudio.google.com/apikey

En Colab, guárdala como Secret (icono de llave 🔑) con el nombre `GOOGLE_API_KEY`, o pégala manualmente abajo.

In [ ]:
import os

try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = None

if not GOOGLE_API_KEY:
    from getpass import getpass
    GOOGLE_API_KEY = getpass("Pega tu GOOGLE_API_KEY: ")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

GEMINI_MODEL = "gemini-3.1-flash-lite"
print("Gemini configurado:", GEMINI_MODEL)


## 2. Base de conocimiento de ejemplo

Sustituye estos documentos por fragmentos reales de la normativa (o carga un PDF en la sección 8).

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content=(
            "El Reglamento de Registro de Consumidores de Combustibles en Bidones "
            "establece que toda persona que requiera adquirir combustibles líquidos "
            "fuera de tanque debe registrarse previamente ante la ANH."
        ),
        metadata={"fuente": "Reglamento RANANHDJUGJN00082025", "articulo": "Art. 1"}
    ),
    Document(
        page_content=(
            "El volumen máximo autorizado para el registro en bidones se determina "
            "según la zona geográfica del solicitante, conforme a los límites "
            "establecidos en la normativa vigente."
        ),
        metadata={"fuente": "Reglamento RANANHDJUGJN00082025", "articulo": "Art. 5"}
    ),
    Document(
        page_content=(
            "La verificación de identidad del solicitante se realiza mediante el "
            "servicio de Ciudadanía Digital, conforme a los lineamientos "
            "AGETIC-RA00272026."
        ),
        metadata={"fuente": "Lineamientos AGETIC RA00272026", "articulo": "Sección 3"}
    ),
    Document(
        page_content=(
            "El Decreto Supremo 5400 regula el consumo de diésel y gasolina, "
            "estableciendo las condiciones bajo las cuales se autoriza el "
            "abastecimiento fuera de las estaciones de servicio."
        ),
        metadata={"fuente": "DS 5400", "articulo": "Art. 3"}
    ),
]

print(f"Documentos de ejemplo: {len(documents)}")


## 3. Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

chunks = text_splitter.split_documents(documents)
print(f"Chunks generados: {len(chunks)}")


## 4. Embeddings (multilingüe, corre en local/CPU)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings listos:", EMBEDDING_MODEL)


## 5. Base vectorial (Chroma)

In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_anh"
)

print("Chunks indexados:", vectorstore._collection.count())


## 6. Retrieval + construcción de contexto con trazabilidad

Cada fragmento recuperado conserva su fuente y artículo, para citarlos en la respuesta.

In [ ]:
def recuperar_contexto(pregunta, k=3):
    docs = vectorstore.similarity_search(pregunta, k=k)

    bloques = []
    for i, doc in enumerate(docs, start=1):
        fuente = doc.metadata.get("fuente", "sin fuente")
        articulo = doc.metadata.get("articulo", "")
        bloques.append(
            f"[Fragmento {i} | {fuente} {articulo}]\n{doc.page_content}"
        )

    contexto = "\n\n".join(bloques)
    return contexto, docs


## 7. Generación con Gemini API (RAG)

In [ ]:
INSTRUCCIONES_SISTEMA = """
Eres un asistente que orienta sobre trámites de la ANH usando exclusivamente el CONTEXTO proporcionado.

REGLAS:
1. Utiliza únicamente la información del CONTEXTO.
2. Si la respuesta no está en el contexto, responde:
   "No encuentro esa información en los documentos proporcionados."
3. No inventes datos ni interpretes la normativa más allá del texto entregado.
4. Cita la fuente y artículo de cada fragmento que utilices.
5. Responde en español, de forma clara y concisa.
6. Aclara que la respuesta es orientativa y no sustituye la interpretación oficial de la ANH.
"""

def generar_con_gemini(system_instruction, user_content, max_tokens=400):
    model = genai.GenerativeModel(
        model_name=GEMINI_MODEL,
        system_instruction=system_instruction
    )
    respuesta = model.generate_content(
        user_content,
        generation_config={"max_output_tokens": max_tokens, "temperature": 0.2}
    )
    return respuesta.text.strip()


def responder_con_rag(pregunta, k=3):
    contexto, docs = recuperar_contexto(pregunta, k=k)

    mensaje_usuario = f"""CONTEXTO:
{contexto}

PREGUNTA:
{pregunta}"""

    respuesta = generar_con_gemini(INSTRUCCIONES_SISTEMA, mensaje_usuario)
    return respuesta, docs


## 8. Probar el RAG

In [ ]:
preguntas_prueba = [
    "¿Quién debe registrarse para consumir combustibles en bidones?",
    "¿Cómo se determina el volumen máximo autorizado?",
    "¿Cómo se verifica la identidad del solicitante?",
    "¿Quién es el director general de la ANH?",
]

for pregunta in preguntas_prueba:
    respuesta, fuentes = responder_con_rag(pregunta, k=2)
    print("=" * 90)
    print("P:", pregunta)
    print("R:", respuesta)
    print("Fuentes:", [f"{d.metadata.get('fuente')} {d.metadata.get('articulo')}" for d in fuentes])


## 9. Notas para migrar esto al backend del proyecto

- Este notebook es para prototipar y validar el pipeline. El componente productivo (Motor de Recuperación Aumentada) debe exponerse como servicio (API REST), no como notebook.
- Reemplaza los documentos de ejemplo por el corpus real segmentado con criterios normativos (por artículo/sección, no solo por caracteres).
- Agrega control de versión de las fuentes en los metadatos (fecha de vigencia, número de norma).
- Considera un umbral mínimo de similitud antes de pasar fragmentos al LLM, para evitar alucinaciones cuando no hay información relevante.
- La clave de Gemini nunca debe quedar hardcodeada en el código del backend: usar variables de entorno/secret manager.